# Classificador de perguntas de compradores do Mercado Livre

Um modelo que lê a pergunta de um comprador e diz de que **tipo** ela é — compatibilidade com o carro, lado da
peça, o que vem no anúncio, medida, outra peça, compra já feita, preço e envio, ou outros —, para quem atende
responder mais rápido, com respostas prontas por tipo.

**Sobre os dados deste repositório:** as perguntas reais são de compradores e da empresa, e não são publicadas.
Os arquivos em `dados/` são um **exemplo inventado**, com o mesmo formato, para o notebook rodar de ponta a
ponta. Os números que você vai ver rodando com eles **não** são os do projeto; os resultados reais estão no
`README.md` e citados ao longo do texto como "nos dados reais".

**Os dois arquivos:**
- `dados/rotulos_exemplo.csv`: perguntas com a etiqueta certa (o **y**) e a coluna `origem`;
- `dados/perguntas_exemplo.csv`: perguntas sem etiqueta, para o Passo 15.

No Colab: na aba **Arquivos** (ícone de pasta, à esquerda), crie uma pasta `dados` e arraste os dois CSVs para
dentro dela.

**De onde vem cada rótulo (coluna `origem`), nos dados reais:**
| origem | quantas | quem rotulou |
|---|---|---|
| `gabarito` | 801 | sugestão de um assistente de IA (Claude), revisada por mim uma por uma |
| `cega_consenso` | 150 | eu e o assistente, **sem ver um ao outro**, com as divergências decididas juntos |
| `claude` | 4.301 | o assistente, seguindo as regras combinadas, sem revisão linha a linha |

Rode uma célula por vez (Shift + Enter) e leia o texto antes de cada uma.

## Passo 1 — Importar as bibliotecas

Biblioteca é um pacote de código pronto que alguém já escreveu e testou. No Colab elas **já vêm instaladas**;
o `import` só as carrega para este notebook usar. Ficam todas aqui no começo para você ver de uma vez do que o
notebook depende.

O `as` dá um apelido curto: em vez de escrever `pandas.read_csv(...)` toda vez, escrevemos `pd.read_csv(...)`.
`pd` e `plt` são os apelidos que todo mundo usa, e você vai encontrá-los em qualquer tutorial.

| biblioteca | apelido | para que serve aqui |
|---|---|---|
| `pandas` | `pd` | ler os CSVs e trabalhar com tabelas (o "Excel" do Python) |
| `matplotlib.pyplot` | `plt` | desenhar os gráficos |
| `re` e `unicodedata` | — | procurar palavras e tirar acentos, na regra de palavras-chave (Passo 4) |
| `json` | — | gravar o modelo num arquivo de texto que o sistema em TypeScript consegue ler (Passo 16) |
| `scikit-learn` (`sklearn`) | — | o machine learning: o TF-IDF, a regressão logística, a separação treino/prova e as notas |

O scikit-learn é grande e dividido em módulos; de cada um importamos só o que vamos usar, pelo nome.
`from sklearn.metrics import accuracy_score` quer dizer "do módulo de métricas, traga a função `accuracy_score`".

A última linha mostra a versão do scikit-learn. Se um número sair um pouco diferente do esperado, a versão é o
primeiro suspeito.

In [ ]:
# Tabelas, números e gráficos
import pandas as pd
import matplotlib.pyplot as plt

# Texto: usados na regra de palavras-chave (Passo 4)
import re
import unicodedata

# Gravar o modelo em arquivo (Passo 16)
import json

# scikit-learn: o machine learning
import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict   # separar treino e prova
from sklearn.feature_extraction.text import TfidfVectorizer                                # texto -> números
from sklearn.linear_model import LogisticRegression                                       # o modelo
from sklearn.pipeline import make_pipeline                                                # juntar os dois numa peça só
from sklearn.metrics import (accuracy_score, recall_score,                                # as notas
                             classification_report, ConfusionMatrixDisplay)

# Só existe no Colab: usado no Passo 15 para baixar o resultado
try:
    from google.colab import files
except ImportError:
    files = None

print('scikit-learn versão', sklearn.__version__)

## Passo 2 — Ler e olhar os dados

`sep=';'` porque o separador é ponto e vírgula; `dtype=str` lê tudo como texto (o número da pergunta não vira
número com casas decimais); `fillna('')` troca célula vazia por texto vazio.

Se aparecer `FileNotFoundError`, um dos dois arquivos não está na pasta `dados`, ou está com outro nome.

In [ ]:
PASTA = 'dados/'   # onde estão os CSVs
dados = pd.read_csv(PASTA + 'rotulos_exemplo.csv', sep=';', dtype=str, encoding='utf-8-sig').fillna('')
todas = pd.read_csv(PASTA + 'perguntas_exemplo.csv', sep=';', dtype=str, encoding='utf-8-sig').fillna('')
print('rotuladas:', dados.shape, '| todas:', todas.shape)
display(dados[['titulo_anuncio', 'pergunta', 'rotulo', 'origem']].head(8))

## Passo 3 — Quantas de cada tipo, e de que origem?

`pd.crosstab` monta uma tabela cruzada: etiqueta nas linhas, origem nas colunas. É **desequilibrado** de
propósito, como nos dados reais: lá, metade das 5.252 perguntas é `compat`, e a etiqueta mais rara
(`preco_envio`) tem 117.

In [ ]:
ordem = ['compat', 'lado', 'conteudo', 'medida', 'outra_peca', 'pedido', 'preco_envio', 'outros']
display(pd.crosstab(dados['rotulo'], dados['origem'], margins=True, margins_name='total').reindex(ordem + ['total']))
dados['rotulo'].value_counts().reindex(ordem).plot(kind='barh', title='Perguntas por etiqueta').invert_yaxis()

## Passo 4 — A régua: a regra de palavras-chave

Procura palavras na pergunta (`serve`, `lado`, `vem com`...) e devolve a primeira categoria
que casar. É o que um programador faria sem ML; o modelo tem de passar disso.

In [ ]:
def sem_acento(s):
    s = unicodedata.normalize('NFKD', str(s).lower())
    return ''.join(c for c in s if not unicodedata.combining(c))

REGRAS = [
    ('pedido',      r'\b(veio|chegou|recebi|enviaram|mandaram|me enviou|comprei e|minha compra|meu pedido|devolu|nao chegou|rastreio|despachad)'),
    ('preco_envio', r'\b(faz (por )?\d|desconto|valor|preco|frete|transportadora|onibus|retirar|retirada|envia para|prazo de entrega|chega quando|nota fiscal|nf\b|pix|boleto|melhor preco|fecha (em|nos))'),
    ('lado',        r'\b(lado|direit|esquerd|motorista|passageiro|par\b|os dois|ambos)'),
    ('medida',      r'\b(\d+ ?mm|medida|diametro|tamanho|comprimento|largura|material|plastico|metal|ferro|cor\b|preta|preto|cinza|original|paralelo|qualidade|rosca|encaixe|pintura|abs\b)'),
    ('conteudo',    r'\b(vem com|acompanha|vem junto|inclui|quantas pecas|quantas unidades|kit vem|vem os|vem as|vem o|vem a|so a peca|completo)'),
    ('outra_peca',  r'\b(tem (para|pra|p/|do|da|de|o|a|só|so)|teria|voces tem|vcs tem|vende (so|separad)|possui|tem disponivel|vende o)'),
    ('compat',      r'\b(serve|servir|compativel|da certo|dá certo|funciona|encaixa|cabe|aplica|vale para|e para o|e do|é do)'),
    ('outros',      r'^(oi|ola|boa (tarde|noite)|bom dia|obrigad)'),
]

def regra(pergunta):
    t = sem_acento(pergunta)
    for etiqueta, padrao in REGRAS:
        if re.search(padrao, t):
            return etiqueta
    return 'outros'

print(regra('Serve no Corsa classic 2011?'))
print(regra('Vem com a braçadeira?'))

## Passo 5 — A prova: quais perguntas ficam escondidas

Esta é a decisão mais importante do notebook. Os rótulos têm **qualidades diferentes**, e a prova tem de usar os
mais confiáveis:

- as **cegas** (dois rotuladores independentes, divergências decididas juntos);
- **20% sorteados do gabarito** (revisados à mão).

Os rotulados só pelo assistente vão **todos para o treino**. Se fossem para a prova, ela mediria "o modelo
concorda com o assistente?", e não "o modelo acerta?".

`stratify` mantém a proporção de cada etiqueta; `random_state=42` fixa o sorteio.

In [ ]:
gab = dados[dados['origem'] == 'gabarito']
gab_treino, gab_prova = train_test_split(gab, test_size=0.2, stratify=gab['rotulo'], random_state=42)
prova = pd.concat([gab_prova, dados[dados['origem'] == 'cega_consenso']])
treino = dados.drop(prova.index)
print('treino:', len(treino), '| prova:', len(prova))
display(prova['rotulo'].value_counts().reindex(ordem).to_frame('na prova'))

## Passo 6 — O modelo, com poucos e com muitos dados

**TF-IDF de pedaços de letras** (`char_wb`, de 2 a 5 letras, para aguentar erro de digitação como "servi",
"corola") e **regressão logística**. O `Pipeline` junta os dois numa peça só.

**`min_df=2`: só entra no vocabulário o pedaço que aparece em pelo menos 2 perguntas.** É uma decisão de
**privacidade**: o vocabulário vai para o sistema da empresa (Passo 16), e com `min_df=1` ele guardava os pedaços
de TODAS as perguntas — inclusive das que aparecem uma vez só. Nos dados reais, em 12 perguntas com CEP ou
telefone, todos os pedaços do número estavam no vocabulário, e encaixando os pedaços dava para remontá-lo. Com
`min_df=2`, nenhuma. Custou 0,6 ponto de acerto e deixou o arquivo com metade do tamanho.

Treinamos duas vezes, e as duas são corrigidas na **mesma prova**:
- só com a parte do gabarito que sobrou no treino (nos dados reais, 640 perguntas);
- com o treino inteiro (nos dados reais, 4.941).

In [ ]:
def novo_modelo(peso=None):
    return make_pipeline(
        TfidfVectorizer(strip_accents='unicode', lowercase=True, analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True,
                        min_df=2),
        LogisticRegression(max_iter=3000, C=10, class_weight=peso),
    )

def nota(nome, chute):
    acc = accuracy_score(prova['rotulo'], chute)
    rec = recall_score(prova['rotulo'], chute, average='macro')
    print(f'{nome:34s} acerto {acc:6.1%}   recall médio por etiqueta {rec:6.1%}')

nota("Sempre 'compat'", ['compat'] * len(prova))
nota('Regra de palavras-chave', prova['pergunta'].map(regra))
m_pequeno = novo_modelo().fit(gab_treino['pergunta'], gab_treino['rotulo'])
nota(f'Modelo, só gabarito ({len(gab_treino)})', m_pequeno.predict(prova['pergunta']))
m_grande = novo_modelo().fit(treino['pergunta'], treino['rotulo'])
nota(f'Modelo, treino inteiro ({len(treino)})', m_grande.predict(prova['pergunta']))

**Como ler as duas notas:**
- **acerto** (acurácia): de todas as perguntas da prova, quantas o modelo acertou. Como metade é `compat`, ela
  esconde quem vai mal nas etiquetas pequenas.
- **recall médio por etiqueta**: calcula o recall de cada etiqueta (de todas as perguntas que são `lado`, quantas
  ele achou) e tira a média, **com o mesmo peso para todas**. Aqui `preco_envio` conta tanto quanto `compat`.
  É a nota que mais subiu com os dados novos.

## Passo 7 — Compensar o desequilíbrio: `class_weight='balanced'`

O modelo aprende errando e corrigindo. Com 2.400 exemplos de `compat` e 100 de `preco_envio` (dados reais), ele
erra `preco_envio` pouco **em número**, e por isso corrige pouco. `class_weight='balanced'` faz cada erro numa
etiqueta rara **pesar mais**, na proporção de quanto ela é rara. É como dar mais pontos na prova para a questão
difícil.

Outra saída seria **jogar fora** parte dos `compat` do treino. Testei as duas; o peso ganhou, e sem perder dado
nenhum.

In [ ]:
m_bal = novo_modelo('balanced').fit(treino['pergunta'], treino['rotulo'])
prova = prova.copy()
prova['y_chapeu'] = m_bal.predict(prova['pergunta'])
nota('Modelo, treino inteiro, sem peso', m_grande.predict(prova['pergunta']))
nota('Modelo, treino inteiro, balanced', prova['y_chapeu'])

## Passo 8 — A matriz de confusão e o relatório por etiqueta

Linha = verdadeiro (y), coluna = o que o modelo chutou (ŷ), diagonal = acertos.

No relatório, `support` é quantas perguntas daquela etiqueta havia na prova. Nos dados reais, `preco_envio` tinha
só 8 na prova: com 8 perguntas, uma a mais ou a menos certa muda o recall em 12 pontos, e o número dessa linha é
um indício, não uma medida.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(prova['rotulo'], prova['y_chapeu'], labels=ordem,
                                        xticks_rotation=45, ax=ax, colorbar=False)
ax.set_xlabel('o modelo chutou (ŷ)'); ax.set_ylabel('verdadeiro (y)')
plt.show()
print(classification_report(prova['rotulo'], prova['y_chapeu'], labels=ordem, zero_division=0))

## Passo 9 — Curva de aprendizado: mais dados ainda ajudariam?

Treinamos com pedaços cada vez maiores do treino e corrigimos sempre a mesma prova. Se a curva ainda sobe no
fim, rotular mais vale a pena. Se ela achatou, o gargalo é outro (a própria definição das etiquetas, por exemplo).

In [ ]:
tamanhos = [int(len(treino) * f) for f in (0.13, 0.26, 0.5)] + [len(treino)]   # nos dados reais: 640, 1.280, 2.500, 4.941
curva = []
for n in tamanhos:
    parte = treino.sample(n, random_state=3) if n < len(treino) else treino
    chute = novo_modelo('balanced').fit(parte['pergunta'], parte['rotulo']).predict(prova['pergunta'])
    curva.append({'perguntas no treino': n,
                  'acerto': accuracy_score(prova['rotulo'], chute),
                  'recall médio': recall_score(prova['rotulo'], chute, average='macro')})
curva = pd.DataFrame(curva).set_index('perguntas no treino')
display(curva.style.format('{:.1%}'))
curva.plot(marker='o', title='Mais perguntas no treino → nota na mesma prova', ylim=(0.5, 1)); plt.show()

## Passo 10 — O teto: quanto concordam duas pessoas?

Nas perguntas cegas, guardamos o rótulo **de cada um antes da conversa** (`cego_pietro` e `cego_claude`). Isso
permite uma comparação que diz muito sobre o modelo:

- nos dados reais, os dois rotuladores concordaram em 84%: nenhum modelo vai acertar 99% quando a própria
  "resposta certa" é discutível em parte das perguntas;
- e o modelo concorda **mais com quem rotulou o treino**. Como a maior parte do treino foi rotulada pelo
  assistente, era esperado que ele concordasse mais com o assistente (90,7%) do que comigo (81,3%). A diferença
  mostra o quanto o modelo aprendeu o **estilo** de um rotulador, e não só a tarefa.

In [ ]:
cega = dados[dados['origem'] == 'cega_consenso'].copy()
cega['modelo'] = m_bal.predict(cega['pergunta'])
pares = [('Pietro', 'Claude', 'cego_pietro', 'cego_claude'),
         ('Modelo', 'Pietro', 'modelo', 'cego_pietro'),
         ('Modelo', 'Claude', 'modelo', 'cego_claude'),
         ('Modelo', 'consenso', 'modelo', 'rotulo')]
for a, b, ca, cb in pares:
    print(f'{a:7s} × {b:9s} concordam {(cega[ca] == cega[cb]).mean():.1%}')

## Passo 11 — Onde o modelo erra?

Os erros da prova, um por um. Muitos são perguntas com dois assuntos, ou continuação de conversa, em que a
fronteira entre etiquetas é discutível mesmo.

In [ ]:
erros = prova[prova['rotulo'] != prova['y_chapeu']]
print(len(erros), 'erros de', len(prova))
display(erros[['titulo_anuncio', 'pergunta', 'rotulo', 'y_chapeu', 'origem']]
        .rename(columns={'rotulo': 'y', 'y_chapeu': 'ŷ'}))

## Passo 12 — Validação cruzada no conjunto inteiro

A prova é pequena (nos dados reais, 311 perguntas), e nas etiquetas pequenas isso é pouco. A **validação cruzada** divide o conjunto inteiro em 5
partes e faz 5 provas, cada pergunta caindo na prova uma vez, e devolve uma nota **por etiqueta** com muito mais
exemplos.

A ressalva: aqui a prova inclui os rótulos do assistente, então ela mede a concordância com o critério combinado, e
tende a dar um pouco mais que o Passo 7. Use para comparar etiquetas entre si, não como a nota oficial.

`cross_val_predict` devolve o chute de cada pergunta feito pelo modelo que **não** a viu no treino.

In [ ]:
dobras = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
chute_cv = cross_val_predict(novo_modelo('balanced'), dados['pergunta'], dados['rotulo'], cv=dobras)
print(classification_report(dados['rotulo'], chute_cv, labels=ordem, zero_division=0))

## Passo 13 — O modelo final

Para usar de verdade, treinamos com **todas** as perguntas rotuladas — a prova já cumpriu o papel dela, e esconder perguntas
agora só deixaria o modelo pior. A nota que vale para ele é a do Passo 7.

In [ ]:
final = novo_modelo('balanced').fit(dados['pergunta'], dados['rotulo'])
for p in ['Serve no Corsa classic 2011?', 'Tem o lado direito?', 'Vem com a braçadeira?',
          'Faz por 50 no pix?', 'Meu pedido não chegou', 'Qual o diâmetro do furo?']:
    print(f'{p:32s} → {final.predict([p])[0]}')

## Passo 14 — Confiança

`predict_proba` dá a probabilidade que o modelo dá a cada etiqueta; a maior delas é a **confiança**. A tabela
mostra a nota da prova separada por faixa de confiança: se o acerto cai muito abaixo de um certo ponto, esse é
o ponto para a tela pedir um olho humano em vez de afirmar a etiqueta.

In [ ]:
prova['confianca'] = m_bal.predict_proba(prova['pergunta']).max(axis=1)
faixas = pd.cut(prova['confianca'], [0, 0.5, 0.7, 0.9, 1.0])
display(prova.assign(acertou=prova['rotulo'] == prova['y_chapeu'])
        .groupby(faixas, observed=True)['acertou'].agg(perguntas='size', acerto='mean')
        .style.format({'acerto': '{:.1%}'}))

## Passo 15 — Usar nas perguntas sem etiqueta e baixar

Classifica todas as perguntas extraídas e responde a pergunta de negócio: **quais anúncios recebem mais perguntas
de compatibilidade?** Anúncio com dezenas de "serve no meu carro?" é anúncio cujo título ou descrição não está
deixando isso claro.

In [ ]:
todas['etiqueta'] = final.predict(todas['pergunta'])
todas['confianca'] = final.predict_proba(todas['pergunta']).max(axis=1).round(2)
display(todas['etiqueta'].value_counts().reindex(ordem).to_frame('perguntas'))
compat = (todas[todas['etiqueta'] == 'compat']
          .groupby(['conta', 'anuncio', 'titulo_anuncio']).size()
          .sort_values(ascending=False).head(15).to_frame('perguntas de compatibilidade'))
display(compat)

todas.to_csv('perguntas_classificadas_v2.csv', sep=';', index=False, encoding='utf-8-sig')
if files:
    files.download('perguntas_classificadas_v2.csv')
else:
    print('Fora do Colab: arquivo salvo como perguntas_classificadas_v2.csv')

## Passo 16 — Exportar o modelo para o sistema em TypeScript

O sistema da empresa roda num Cloudflare Worker, em TypeScript, não em Python. Então o modelo vai para lá como
um arquivo de texto, o `modelo_perguntas.json`, com o que o `fit` do Passo 13 aprendeu:

| no JSON | de onde vem | o que é |
|---|---|---|
| `pedacos` | `vocabulary_`, `idf_` e `coef_` | para cada pedaço de letras: a raridade e um peso por etiqueta |
| `base` | `intercept_` | um ajuste fixo por etiqueta |
| `etiquetas` | `classes_` | a ordem das etiquetas, que é a ordem dos pesos |
| `nota_na_prova` | Passo 7 | a nota deste modelo, para ficar registrada junto dele |

Os números vão com **6 algarismos significativos** em vez de 17: o arquivo cai pela metade, e a diferença nas
probabilidades fica abaixo de 0,0001%.

O segundo arquivo, `conferencia_perguntas.json`, é a **prova de que o TypeScript faz a mesma conta**: algumas
perguntas e a resposta que o Python deu para cada uma. Do lado do TypeScript, um teste passa as mesmas perguntas
pela função e exige a mesma etiqueta e a mesma probabilidade. Nos dados reais, conferido nas 5.462 perguntas:
nenhuma etiqueta trocada.

As perguntas da conferência são **inventadas**, de propósito: texto de comprador não vai para repositório. Elas
cobrem as 8 etiquetas e os casos difíceis de copiar entre linguagens: acento, cedilha, maiúscula, espaço duplo,
tabulação, emoji e pergunta vazia.

In [ ]:
vetorizador, regressao = final[0], final[1]
nomes = vetorizador.get_feature_names_out()
idf = vetorizador.idf_
pesos = regressao.coef_          # uma linha por etiqueta, uma coluna por pedaço

def curto(v):
    return float(f'{v:.6g}')     # 6 algarismos significativos

modelo_json = {
    'versao': pd.Timestamp.now().strftime('%Y-%m-%d'),
    'sklearn': sklearn.__version__,
    'treinado_com': len(dados),
    'nota_na_prova': {
        'acerto': round(accuracy_score(prova['rotulo'], prova['y_chapeu']), 4),
        'recall_medio': round(recall_score(prova['rotulo'], prova['y_chapeu'], average='macro'), 4),
    },
    'ngram': list(vetorizador.ngram_range),
    'etiquetas': regressao.classes_.tolist(),
    'base': [curto(b) for b in regressao.intercept_],
    'pedacos': {nome: [curto(idf[i])] + [curto(w) for w in pesos[:, i]] for i, nome in enumerate(nomes)},
}
with open('modelo_perguntas.json', 'w', encoding='utf-8') as arquivo:
    json.dump(modelo_json, arquivo, ensure_ascii=False, separators=(',', ':'))

CONFERENCIA = [
    'Serve no Corsa Classic 2011?', 'dá certo no HB20 2019 1.0?', 'SERVE NA HILUX SRV 2015???',
    'esse spoiler cabe no onix 2020 sem escape aparente?', 'compativel com fox 2012',
    'é o lado do motorista?', 'tem o lado direito?', 'preciso do esquerdo e do direito', 'Qual lado é esse?',
    'vem com a braçadeira?', 'acompanha os parafusos?', 'é o par ou só uma unidade?', 'quantas presilhas vem no kit?',
    'qual o diâmetro do furo?', 'é de plástico ou metal?', 'é original ou paralela?', 'a manopla é de rosca?',
    'tem para o civic 2008?', 'vocês tem na cor preta?', 'vende só os parafusos?', 'tem o traseiro também?',
    'meu pedido não chegou', 'comprei o direito e veio o esquerdo', 'quero devolver', 'quando vão despachar minha compra?',
    'faz por 50 no pix?', 'qual o valor do frete para Manaus?', 'emite nota fiscal?', 'consegue um desconto no par?',
    'boa tarde', 'obrigado!', 'ok vou comprar', '???',
    '', '   ', 'Serve  no\tGol   G5?', 'ÉÉÉ ÇÇÇ ãõ', 'serve 😀👍 no gol', 'ﬁat uno', 'Ｈｉｌｕｘ 2015',
    'S E R V E', '12345-67890 serve?', 'parabarro ' * 40,
]
probs = final.predict_proba(CONFERENCIA)
conferencia = [
    {'pergunta': p,
     'etiqueta': regressao.classes_[k.argmax()],
     'probabilidades': dict(zip(regressao.classes_.tolist(), k.round(8).tolist()))}
    for p, k in zip(CONFERENCIA, probs)
]
with open('conferencia_perguntas.json', 'w', encoding='utf-8') as arquivo:
    json.dump({'versao': modelo_json['versao'], 'casos': conferencia}, arquivo, ensure_ascii=False, indent=1)

print(f"modelo: {len(nomes)} pedaços, nota na prova {modelo_json['nota_na_prova']}")
print('conferência:', len(conferencia), 'perguntas')
if files:
    files.download('modelo_perguntas.json')
    files.download('conferencia_perguntas.json')
else:
    print('Fora do Colab: arquivos salvos na pasta.')